# Imports

In [18]:
import os              
import pandas as pd  

# Define country and parameters

In [19]:
# Select target country
country = 'Colombia'

# Set country-specific parameters: ISO codes and buffer size (in meters)
if country == 'Argentina':
    country_short = 'ARG'   # ISO 3-letter code
    country_code = 'AR'     # ISO 2-letter code
elif country == 'Chile':
    country_short = 'CHL'
    country_code = 'CL'
elif country == 'Colombia':
    country_short = 'COL'
    country_code = 'CO'
# Uncomment the following if Mexico is to be included in the analysis
# elif country == 'Mexico':
#     country_short = 'MEX'
#     country_code = 'MX'

# Set working directory

In [20]:
# Define working directory path
wd = (
#     '/your/path/to/working/directory/'
    '/Users/carmen/Library/CloudStorage/OneDrive-TheUniversityofLiverpool/research/latin-mobility-covid'
)

# Baseline Facebook population

In [21]:
df_bl = pd.read_csv(
    os.path.join(wd, 'data', 'outputs', country_short, 'baseline', 'baseline_pop.csv')
)

present = 0
incomplete = 0
missing = 0
total = 0

for i in range(6):
    col = str(i)
    present += len(df_bl[df_bl[col] > 0])
    incomplete += len(df_bl[df_bl[col] == -999])
    total += len(df_bl)
    missing += len(df_bl) - len(df_bl[df_bl[col] > 0]) - len(df_bl[df_bl[col] == -999])

print(present, total, total - present, (total - present) / total * 100)

57127 287382 230255 80.12158033558121


# Baseline Facebook movement

In [22]:
long_candidates = [
    os.path.join(
        wd, 'data', 'outputs', country_short, 'baseline',
        'movcell-baseline-imput-mov-dist-with-exo-var-subunit0-sample.csv'
    ),
    os.path.join(
        wd, 'data', 'outputs', country_short, 'baseline',
        'movcell-baseline-imput-mov-dist-with-exo-var-sample.csv'
    ),
]

long_file = next((p for p in long_candidates if os.path.exists(p)), None)
if long_file is None:
    raise FileNotFoundError('No long-format baseline movement imputation file found.')

df_bl = pd.read_csv(long_file)

if 'was_missing' not in df_bl.columns:
    df_bl['was_missing'] = df_bl['y'] == -1
if 'model_imputed' not in df_bl.columns:
    df_bl['model_imputed'] = df_bl['was_missing']
if 'subunit_rule_applied' not in df_bl.columns:
    df_bl['subunit_rule_applied'] = False

df_bl['count_as_model_imputed'] = df_bl['model_imputed'] & (~df_bl['subunit_rule_applied'])

pair_summary = (
    df_bl.groupby(['O', 'D'], as_index=False)
    .agg(
        any_missing=('was_missing', 'max'),
        all_missing=('was_missing', 'min'),
        any_model_imputed=('count_as_model_imputed', 'max')
    )
)

summary = {
    'total_records': int(len(pair_summary)),
    'missing_records': int(pair_summary['any_missing'].sum()),
    'pct_total_imputed': float((pair_summary['all_missing'] & pair_summary['any_model_imputed']).mean() * 100)
}
print(summary)


{'total_records': 108867, 'missing_records': 106140, 'pct_total_imputed': 90.5563669431508}


# Crisis Facebook population

In [23]:
# Load CSV and drop unnecessary column
file_path = os.path.join(
    wd, 'data', 'outputs', country_short, 'evo', 'pop_evo_popcell_raw.csv'
)
df_cr = pd.read_csv(file_path).drop(columns=['Unnamed: 0'])

present = 0
incomplete = 0
missing = 0
total = 0

# Iterate through all columns except the first one
for column in df_cr.columns[1:]:
    present += len(df_cr[df_cr[column] > 0])
    incomplete += len(df_cr[df_cr[column] == -999])
    total += len(df_cr)
    missing += (
        len(df_cr)
        - len(df_cr[df_cr[column] > 0])
        - len(df_cr[df_cr[column] == -999])
    )

print(present, total, total - present, (total - present) / total * 100)

7175914 38374072 31198158 81.30009762841952


# Crisis Facebook movement

In [24]:
file_path = os.path.join(
    wd, 'data', 'outputs', country_short, 'evo', 'mov_evo_dist_raw.csv'
)
df_cr = pd.read_csv(file_path).drop(columns=['Unnamed: 0'], errors='ignore')

data_cols = list(df_cr.columns[2:])
any_positive = (df_cr[data_cols] > 0).any(axis=1)
any_missing = ~(df_cr[data_cols] > 0).all(axis=1)
model_imputed_pair = ~any_positive

summary = {
    'total_records': int(len(df_cr)),
    'missing_records': int(any_missing.sum()),
    'pct_total_imputed': float(model_imputed_pair.mean() * 100)
}
print(summary)


{'total_records': 116951, 'missing_records': 116951, 'pct_total_imputed': 95.65801061983224}
